In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 

path = os.path.join("..", "data", "loan_eda.csv")

try:
    df=pd.read_csv(path)
    print(df.shape)
    print(df.columns.tolist())

except FileNotFoundError:
    print(f"Error: Could not find the file at '{csv_path}'.")    

(148670, 32)
['loan_limit', 'gender', 'approved_in_advance', 'loan_type', 'loan_purpose', 'credit_worthiness', 'open_credit', 'business_or_commercial', 'loan_amount', 'rate_of_interest', 'interest_rate_spread', 'upfront_charges', 'term', 'negative_amortization', 'interest_only', 'lump_sum_payment', 'property_value', 'construction_type', 'occupancy_type', 'secured_by', 'total_units', 'income', 'credit_type', 'credit_score', 'co_applicant_credit_type', 'age', 'submission_of_application', 'loan_to_value_ratio', 'region', 'security_type', 'status', 'debt_to_income_ratio']


In [2]:
# Leakage: not available at application time, or fake signal
df = df.drop(columns=[
    'rate_of_interest',      # set by lender post-approval
    'interest_rate_spread',  # derived from rate_of_interest
    'upfront_charges',       # calculated after application submitted
    'credit_type',           # EQUI = 100% default across 15,298 rows → label leak
], errors='ignore')
print(f"Shape: {df.shape}")

Shape: (148670, 28)


In [3]:
# Near-constant (>99% one value, see 01 dominance analysis) + tiny artifact
df = df.drop(columns=[
    'security_type',       # ~99.9% 'direct'
    'open_credit',         # ~99% one value
    'credit_worthiness',   # heavily skewed to 'l1'
    'construction_type',   # near-constant
    'secured_by',          # 'land' = 33 rows at 100% default → artifact
], errors='ignore')
print(f"Shape: {df.shape}")
print(df.columns.tolist())

Shape: (148670, 23)
['loan_limit', 'gender', 'approved_in_advance', 'loan_type', 'loan_purpose', 'business_or_commercial', 'loan_amount', 'term', 'negative_amortization', 'interest_only', 'lump_sum_payment', 'property_value', 'occupancy_type', 'total_units', 'income', 'credit_score', 'co_applicant_credit_type', 'age', 'submission_of_application', 'loan_to_value_ratio', 'region', 'status', 'debt_to_income_ratio']


In [4]:
# Flags must be created now — imputation in notebook 03 will erase the NaNs
df['income_missing']  = df['income'].isnull().astype(int)
df['dtir_missing']    = df['debt_to_income_ratio'].isnull().astype(int)
df['propval_missing'] = df['property_value'].isnull().astype(int)

# Confirm each flag carries signal
for f in ['income_missing', 'dtir_missing', 'propval_missing']:
    print(f, df.groupby(f)['status'].mean().round(3).to_dict())

income_missing {0: 0.254, 1: 0.135}
dtir_missing {0: 0.163, 1: 0.676}
propval_missing {0: 0.161, 1: 1.0}


In [5]:
# NaNs must still be present — imputation happens in notebook 03 after the split
print("Columns still containing NaNs:")
print(df.isnull().sum()[df.isnull().sum() > 0])

df.to_csv('../data/loan_features.csv', index=False)
print(f"\nSaved loan_features.csv: {df.shape}")
print(df.columns.tolist())

Columns still containing NaNs:
loan_limit                    3344
approved_in_advance            908
loan_purpose                   134
term                            41
negative_amortization          121
property_value               15098
income                        9150
age                            200
submission_of_application      200
loan_to_value_ratio          15098
debt_to_income_ratio         24121
dtype: int64

Saved loan_features.csv: (148670, 26)
['loan_limit', 'gender', 'approved_in_advance', 'loan_type', 'loan_purpose', 'business_or_commercial', 'loan_amount', 'term', 'negative_amortization', 'interest_only', 'lump_sum_payment', 'property_value', 'occupancy_type', 'total_units', 'income', 'credit_score', 'co_applicant_credit_type', 'age', 'submission_of_application', 'loan_to_value_ratio', 'region', 'status', 'debt_to_income_ratio', 'income_missing', 'dtir_missing', 'propval_missing']
